# Analysing Kp Forecast Calibration

*Reproducible research notebook from [Flarient](https://flarient.com) — the space weather intelligence platform.*

**About this notebook:** This notebook is part of the [Flarient Research Notebooks](https://github.com/flarientglobal/flarient-notebooks) collection. It uses public data from NOAA SWPC, NASA, and the Flarient API.


## 1. Introduction

Forecast calibration measures how well predicted probabilities match observed frequencies. A well-calibrated forecast of 70% should be correct 70% of the time. In this notebook, we'll analyse Kp forecast calibration using public data.


In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Fetching Historical Kp Data

We'll fetch Kp data from NOAA SWPC to build a historical dataset.


In [ ]:
# Fetch Kp data
kp_url = "https://services.swpc.noaa.gov/json/planetary_k_index_1m.json"
response = requests.get(kp_url, timeout=30)
kp_data = response.json()

df = pd.DataFrame(kp_data)
df['time_tag'] = pd.to_datetime(df['time_tag'])
df['kp'] = df['kp'].astype(float)
df = df.set_index('time_tag').sort_index()

print(f"Kp data: {len(df)} readings from {df.index.min()} to {df.index.max()}")


## 3. Building a Simple Forecast

We'll create a simple persistence forecast (tomorrow's Kp = today's Kp) and evaluate its calibration.


In [ ]:
# Create daily max Kp
daily_kp = df['kp'].resample('D').max()

# Persistence forecast: predict tomorrow's max Kp = today's max Kp
forecasts = daily_kp.shift(1).dropna()
actuals = daily_kp[forecasts.index]

# Calculate forecast errors
errors = forecasts - actuals
mae = np.abs(errors).mean()
rmse = np.sqrt((errors ** 2).mean())

print(f"Persistence Forecast Performance:")
print(f"  MAE: {mae:.2f}")
print(f"  RMSE: {rmse:.2f}")
print(f"  Bias: {errors.mean():.2f} (positive = overestimate)")


## 4. Calibration Analysis

A calibration plot compares predicted vs observed frequencies. For Kp, we'll bin forecasts and check if the observed Kp matches.


In [ ]:
# Bin forecasts into ranges and calculate observed frequency
bins = [0, 2, 3, 4, 5, 6, 7, 9]
labels = ['0-2', '2-3', '3-4', '4-5', '5-6', '6-7', '7-9']
forecasts_binned = pd.cut(forecasts, bins=bins, labels=labels, right=False)
actuals_binned = pd.cut(actuals, bins=bins, labels=labels, right=False)

# Calibration table
calibration = pd.DataFrame({
    'forecast_bin': labels,
    'count': [forecasts_binned[fcasts == l].count() for l in labels for fcasts in [forecasts_binned]],
    'observed_mean_kp': [actuals[forecasts_binned == l].mean() for l in labels],
    'forecast_mean_kp': [forecasts[forecasts_binned == l].mean() for l in labels],
}).dropna()

print("Calibration Table:")
print(calibration.to_string(index=False))


## 5. Calibration Plot

The ideal calibration plot has points on the diagonal line.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(calibration['forecast_mean_kp'], calibration['observed_mean_kp'], 
           s=calibration['count'] * 5, alpha=0.7, color='#6366f1')
ax.plot([0, 9], [0, 9], 'w--', alpha=0.3, label='Perfect calibration')
ax.set_xlabel('Forecast Kp')
ax.set_ylabel('Observed Kp')
ax.set_title('Kp Forecast Calibration (Persistence Model) — Source: NOAA SWPC')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('kp_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Comparing with Flarient Forecasts

Flarient provides AI-enhanced forecasts. Visit [flarient.com](https://flarient.com) to see live forecasts and compare with this persistence baseline.

The [Flarient Event Ledger](https://github.com/flarientglobal/flarient-event-ledger) provides a public, versioned record of Flarient's forecasts for independent verification.


## 7. Conclusion

This notebook showed how to:
1. Build a persistence forecast baseline
2. Calculate forecast errors (MAE, RMSE, bias)
3. Create a calibration plot
4. Identify systematic biases in forecasting

For production-quality forecasts, visit [flarient.com](https://flarient.com).


---

## About Flarient

[Flarient](https://flarient.com) is a space weather intelligence platform providing real-time data, forecasts, and community-driven observations. Visit [flarient.com](https://flarient.com) for live space weather conditions, aurora forecasts, and more.

## License

MIT — This notebook is open source. [View on GitHub](https://github.com/flarientglobal/flarient-notebooks).
